# 01 - Tool Engineering with Pydantic

## Scenario: Northstar Refunds

Northstar's AI agents need the ability to issue refunds for failed checkouts. But simply giving an LLM a tool like `issue_refund(amount: int, user_id: str)` is a massive security risk.

Tool Engineering is the practice of designing safe, typed, and verifiable tool schemas. In this lab, we will use `pydantic` to strictly define the tool inputs, enforce validation, and implement the "Prepare Action" pattern to ensure a human approves the refund before execution.

In [1]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

# 1. Defining the strict schema with Pydantic
class PrepareRefundArgs(BaseModel):
    user_id: str = Field(..., description="The exact 12-character alphanumeric Northstar user ID.")
    amount_cents: int = Field(..., gt=0, le=10000, description="Refund amount in cents. Max $100.00.")
    reason: Literal["checkout_failure", "duplicate_charge", "support_discretion"] = Field(
        ..., description="The approved reason category for the refund."
    )
    justification: str = Field(..., min_length=10, description="Detailed explanation of why this refund is necessary.")

# 2. Tool implementation (Prepare Action Pattern)
def prepare_refund(args: PrepareRefundArgs) -> str:
    """
    Instead of actually moving money, this tool prepares a transaction
    and leaves it in a 'pending_approval' state.
    """
    print(f"[Tool Executed] Preparing refund of {args.amount_cents} cents for {args.user_id}...")
    print(f"  Reason: {args.reason}\n  Justification: {args.justification}")
    
    # In a real system, this would insert a row into a database.
    transaction_id = f"REF-{args.user_id[:4]}-8829"
    return f"Refund prepared successfully. Transaction ID: {transaction_id}. Status: PENDING_HUMAN_APPROVAL"


## 1. Why Pydantic?

Using raw JSON schemas (like in the previous module) is error-prone. `Pydantic` allows you to express complex validation logic natively in Python. The LLM will still receive a JSON schema, but Pydantic handles the type casting and boundary checking (e.g. `le=10000` enforces a $100 max refund) automatically before your business logic runs.

In [2]:
# Let's see what happens if the LLM hallucinates an invalid reason or a massive amount.

invalid_llm_output = {
    "user_id": "user_12345678",
    "amount_cents": 500000,  # $5000!
    "reason": "customer_was_angry", # Not an allowed literal
    "justification": "They yelled at me."
}

try:
    # We attempt to cast the LLM's raw JSON output to our Pydantic model
    args = PrepareRefundArgs(**invalid_llm_output)
    result = prepare_refund(args)
except ValidationError as e:
    print("Validation Error Caught! The LLM made a mistake:\n")
    print(e)


Validation Error Caught! The LLM made a mistake:

2 validation errors for PrepareRefundArgs
amount_cents
  Input should be less than or equal to 10000 [type=less_than_equal, input_value=500000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
reason
  Input should be 'checkout_failure', 'duplicate_charge' or 'support_discretion' [type=literal_error, input_value='customer_was_angry', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 2. Returning Validation Errors to the LLM

When a validation error occurs, you shouldn't just crash the application. You should catch the error and **return the error message to the LLM as a tool observation**. The LLM can read the error (e.g., "Input should be less than or equal to 10000") and self-correct on the next step.

In [3]:
def safe_tool_executor(raw_args: dict) -> str:
    try:
        args = PrepareRefundArgs(**raw_args)
        return prepare_refund(args)
    except ValidationError as e:
        # We return the exact error message to the LLM so it knows how to fix its mistake.
        return f"TOOL ERROR: Invalid arguments.\n{str(e)}\nPlease correct your arguments and try again."

# Simulated loop correction
observation = safe_tool_executor(invalid_llm_output)
print("Observation sent back to LLM:\n")
print(observation)


Observation sent back to LLM:

TOOL ERROR: Invalid arguments.
2 validation errors for PrepareRefundArgs
amount_cents
  Input should be less than or equal to 10000 [type=less_than_equal, input_value=500000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
reason
  Input should be 'checkout_failure', 'duplicate_charge' or 'support_discretion' [type=literal_error, input_value='customer_was_angry', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Please correct your arguments and try again.


## 3. Tool Descriptions ARE Prompt Engineering

Notice how we didn't just write `user_id: str`. We wrote `user_id: str = Field(..., description="...")`. 
The descriptions inside your Pydantic fields are often more important than your main system prompt. They tell the LLM exactly how to format the data and when to use the tool.

## Watch For

- **Broad Inputs**: Allowing `reason: str` instead of a constrained `Literal` forces the LLM to guess what your API expects.
- **Side-Effect Tools**: Building a tool that executes an irreversible action (like `transfer_money`) instead of preparing it for approval.
- **Silent Failures**: Catching validation errors but returning "Error" to the LLM instead of the actual Pydantic validation trace.

## Checkpoint

**1. What is the primary benefit of the "Prepare Action" pattern?**
- A) It runs faster than a standard action.
- B) It prevents the LLM from executing irreversible side-effects by requiring human authorization.
- C) It uses less tokens.
- D) It bypasses Pydantic validation.

**2. Why should we return Pydantic `ValidationError` strings directly to the LLM?**
- A) It causes the LLM to crash safely.
- B) It allows the LLM to read the exact constraint it violated and self-correct.
- C) It saves database space.
- D) We shouldn't; we should hide errors from the LLM for security.
